# Praktikum 07 – Vektordatenbanken und RAG-Experimente
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Einen reproduzierbaren Dokumentkorpus indexieren, Vektor- und Hybridsuche vergleichen und RAG-Antworten auf Faktentreue pruefen.

**Zielprodukt nach 90 Minuten:**
1. Ein Chroma-Korpus mit mehreren Themenclustern und Metadaten.
2. Ein Vergleich von reiner Vektorsuche gegen Hybridsuche mit derselben Benchmark.
3. Eine kurze RAG-Auswertung mit dokumentierter Trefferbasis.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "chromadb": ("chromadb", "1.1.1"),
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import chromadb
import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


def model_is_available(requested_name, available_names):
    candidates = {requested_name, f"{requested_name}:latest"}
    return any(candidate in available_names for candidate in candidates)


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
for model_name in [MODEL, EMBED_MODEL]:
    subprocess.check_call(["ollama", "pull", model_name], env=env)

payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
missing_models = [
    model_name
    for model_name in [MODEL, EMBED_MODEL]
    if not model_is_available(model_name, available_models)
]
if missing_models:
    raise RuntimeError(f"Missing local Ollama models: {', '.join(missing_models)}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Models:", MODEL, ",", EMBED_MODEL)
print("Available local models:", ", ".join(available_models))

## Teil 1 – Reproduzierbaren RAG-Korpus aufbauen (25 min)
Wir erzeugen einen kleinen, aber thematisch gemischten Dokumentkorpus mit Metadaten.

**Pflichtschritte:**
- Indexiere mehrere kurze Dokumente aus den Themen Programmierung, ML und Sicherheit.
- Vergib pro Dokument eindeutige IDs und Metadaten.
- Pruefe danach, ob der Korpus erwartungsgemaess in Chroma liegt.

**Soll-Ergebnis:** ein frischer Vektorspeicher mit nachvollziehbarer Dokumentstruktur.

In [ ]:
course_records = [
    {"doc_id": "prog_python", "topic": "programmierung", "level": "intro", "document": "Python ist eine interpretierte Sprache mit sehr grossem Oekosystem fuer Datenanalyse und Automatisierung."},
    {"doc_id": "prog_cpp", "topic": "programmierung", "level": "advanced", "document": "C++ ist eine kompilierte Sprache mit direktem Speicherzugriff und hoher Laufzeitkontrolle."},
    {"doc_id": "prog_rust", "topic": "programmierung", "level": "advanced", "document": "Rust setzt auf Ownership und Borrowing, um Speichersicherheit ohne Garbage Collector zu erreichen."},
    {"doc_id": "prog_java", "topic": "programmierung", "level": "intro", "document": "Java wird typischerweise auf der JVM ausgefuehrt und ist in grossen Unternehmensanwendungen weit verbreitet."},
    {"doc_id": "prog_types", "topic": "programmierung", "level": "intro", "document": "Statische Typisierung erkennt viele Fehler vor der Laufzeit, dynamische Typisierung ist oft flexibler beim Prototyping."},
    {"doc_id": "ml_embedding", "topic": "ml", "level": "intro", "document": "Embeddings bilden Texte als Vektoren ab, sodass semantisch aehnliche Inhalte im Vektorraum nahe beieinander liegen."},
    {"doc_id": "ml_transformer", "topic": "ml", "level": "advanced", "document": "Transformer verarbeiten Sequenzen mit Self-Attention und koennen dadurch lange Abhaengigkeiten modellieren."},
    {"doc_id": "ml_chunking", "topic": "ml", "level": "intro", "document": "Zu grosse Chunks verschlechtern oft Retrieval-Praezision, zu kleine Chunks verlieren wichtigen Kontext."},
    {"doc_id": "ml_rag", "topic": "ml", "level": "intro", "document": "Retrieval Augmented Generation kombiniert externe Dokumente mit einem Sprachmodell, um Antworten besser am Kontext zu verankern."},
    {"doc_id": "ml_faithfulness", "topic": "ml", "level": "advanced", "document": "Faithfulness beschreibt, ob eine Antwort vollstaendig durch den bereitgestellten Kontext gedeckt ist."},
    {"doc_id": "sec_prompt_injection", "topic": "sicherheit", "level": "advanced", "document": "Prompt Injection versucht, die urspruenglichen Anweisungen eines Systems durch manipulative Nutzereingaben zu ueberschreiben."},
    {"doc_id": "sec_moderation", "topic": "sicherheit", "level": "intro", "document": "Ein Moderations-Layer prueft Eingaben oder Ausgaben auf unerwuenschte Inhalte, bevor ein Modell antwortet."},
    {"doc_id": "sec_dlp", "topic": "sicherheit", "level": "intro", "document": "Data Loss Prevention maskiert oder blockiert sensible Daten wie Kreditkartennummern oder personenbezogene Informationen."},
    {"doc_id": "db_vector", "topic": "datenbanken", "level": "intro", "document": "Vektordatenbanken speichern numerische Repräsentationen und beschleunigen Aehnlichkeitssuche ueber viele Dokumente."},
    {"doc_id": "db_relational", "topic": "datenbanken", "level": "intro", "document": "Relationale Datenbanken sind stark bei strukturierten Tabellen, Joins und transaktionalen Workloads."},
    {"doc_id": "db_metadata", "topic": "datenbanken", "level": "advanced", "document": "Metadatenfilter schraenken die Kandidatenmenge vor oder waehrend des Retrievals ein und verbessern oft die Treffergenauigkeit."},
]


def embed_text(text):
    return ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"]


client = chromadb.PersistentClient(path="./rag_storage")
collection_name = "course_docs"
existing_collections = {item.name for item in client.list_collections()}
if collection_name in existing_collections:
    client.delete_collection(collection_name)
collection = client.create_collection(name=collection_name)

collection.add(
    ids=[record["doc_id"] for record in course_records],
    documents=[record["document"] for record in course_records],
    embeddings=[embed_text(record["document"]) for record in course_records],
    metadatas=[
        {
            "doc_id": record["doc_id"],
            "topic": record["topic"],
            "level": record["level"],
        }
        for record in course_records
    ],
)

topic_counts = {}
for record in course_records:
    topic_counts[record["topic"]] = topic_counts.get(record["topic"], 0) + 1

print(f"Anzahl Dokumente in der Datenbank: {collection.count()}")
print("Verteilung nach Themen:")
for topic, count in sorted(topic_counts.items()):
    print(f"- {topic}: {count}")

## Teil 2 – Vektorsuche gegen Hybridsuche vergleichen (30 min)
Wir verwenden dieselben Fragen fuer zwei Retrieval-Varianten und vergleichen die Top-3-Treffer.

**Pflichtschritte:**
- Implementiere eine reine Vektorsuche.
- Ergaenze ein einfaches hybrides Re-Ranking mit Lexik-Signal.
- Messe danach fuer feste Fragen, ob das erwartete Dokument in den Top-3 landet.

**Soll-Ergebnis:** eine kleine Benchmark-Tabelle fuer beide Verfahren.

In [ ]:
import re

STOPWORDS = {"was", "ist", "der", "die", "das", "ein", "eine", "und", "mit", "fuer", "bei", "auf"}


def vector_search(query, top_k=3, where=None):
    request = {
        "query_embeddings": [embed_text(query)],
        "n_results": min(top_k, len(course_records)),
        "include": ["documents", "metadatas", "distances"],
    }
    if where is not None:
        request["where"] = where
    result = collection.query(**request)
    return [
        {
            "document": document,
            "metadata": metadata,
            "distance": distance,
        }
        for document, metadata, distance in zip(
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0],
        )
    ]


def lexical_score(query, document):
    query_terms = [token for token in re.findall(r"\w+", query.lower()) if token not in STOPWORDS]
    document_terms = set(re.findall(r"\w+", document.lower()))
    return sum(1 for token in query_terms if token in document_terms)


def hybrid_search(query, top_k=3, where=None):
    candidates = vector_search(query, top_k=min(8, len(course_records)), where=where)
    ranked = []
    for rank, item in enumerate(candidates, start=1):
        lex_score = lexical_score(query, item["document"])
        vector_bonus = max(0, 8 - rank)
        ranked.append({**item, "score": lex_score * 10 + vector_bonus})
    ranked.sort(key=lambda item: item["score"], reverse=True)
    return ranked[:top_k]


demo_query = "Warum sind Embeddings fuer semantische Suche hilfreich?"
print("Vektorsuche")
for item in vector_search(demo_query, top_k=3):
    print(f"- {item['metadata']['doc_id']}: {item['document']}")

print("\nHybridsuche")
for item in hybrid_search(demo_query, top_k=3):
    print(f"- {item['metadata']['doc_id']}: {item['document']}")

In [ ]:
RETRIEVAL_BENCHMARK = [
    {"query": "Welche Sprache nutzt Ownership fuer Speichersicherheit?", "expected_doc_id": "prog_rust"},
    {"query": "Welche Komponente prueft Eingaben vor dem Modell?", "expected_doc_id": "sec_moderation"},
    {"query": "Wofuer nutzt man Embeddings in RAG?", "expected_doc_id": "ml_embedding"},
    {"query": "Welche Datenbank ist gut fuer Joins und Transaktionen?", "expected_doc_id": "db_relational"},
    {"query": "Was bedeutet Faithfulness bei RAG?", "expected_doc_id": "ml_faithfulness"},
]


def evaluate_search(search_fn, label):
    rows = []
    for case in RETRIEVAL_BENCHMARK:
        results = search_fn(case["query"], top_k=3)
        retrieved_ids = [item["metadata"]["doc_id"] for item in results]
        rows.append(
            {
                "query": case["query"],
                "expected": case["expected_doc_id"],
                "top3": ", ".join(retrieved_ids),
                "hit": "JA" if case["expected_doc_id"] in retrieved_ids else "NEIN",
            }
        )
    print(label)
    for row in rows:
        print(f"- {row['hit']} | {row['expected']} | {row['top3']}")
    hit_rate = sum(row["hit"] == "JA" for row in rows) / len(rows)
    print(f"Trefferquote Top-3: {hit_rate:.2f}\n")
    return rows


vector_rows = evaluate_search(vector_search, "Vektorsuche")
hybrid_rows = evaluate_search(hybrid_search, "Hybridsuche")

## Teil 3 – Kleine RAG-Auswertung mit Faithfulness-Check (25 min)
Wir beantworten feste Fragen mit den gefundenen Dokumenten und pruefen danach die Faktentreue.

**Pflichtschritte:**
- Nutze die Hybridsuche, um Kontext fuer jede Frage zusammenzustellen.
- Lasse das Modell kurz antworten und speichere die verwendeten Dokument-IDs.
- Fuehre danach einen Faithfulness-Check mit JA oder NEIN durch.

**Soll-Ergebnis:** eine nachvollziehbare Antwort pro Frage inklusive Quellenbasis.

In [ ]:
def build_context(query, top_k=3, where=None):
    items = hybrid_search(query, top_k=top_k, where=where)
    blocks = [f"[{item['metadata']['doc_id']}] {item['document']}" for item in items]
    return items, "\n\n".join(blocks)


def rag_with_check(query, where=None):
    items, context = build_context(query, top_k=3, where=where)
    prompt = (
        "Beantworte die Frage ausschliesslich mit Informationen aus dem Kontext. "
        "Wenn die Information fehlt, antworte exakt mit Unbekannt. "
        "Nenne am Satzende in Klammern die genutzten Dokument-IDs.\n\n"
        f"Kontext:\n{context}\n\nFrage: {query}"
    )
    answer = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 80},
        keep_alive="10m",
    )["message"]["content"].strip()
    if not answer:
        raise RuntimeError("Das Modell hat keine Antwort erzeugt.")

    check_prompt = (
        "Pruefe, ob die Antwort vollstaendig durch den Kontext gedeckt ist. "
        "Antworte exakt mit JA oder NEIN.\n\n"
        f"Kontext:\n{context}\n\nAntwort: {answer}"
    )
    validity = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": check_prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 8, "stop": ["\n"]},
        keep_alive="10m",
    )["message"]["content"].strip()

    return {
        "answer": answer,
        "validity": validity,
        "doc_ids": [item["metadata"]["doc_id"] for item in items],
    }


RAG_CASES = [
    "Warum sind Embeddings fuer Retrieval nuetzlich?",
    "Welche Aufgabe hat ein Moderations-Layer?",
    "Welche Sprache laeuft typischerweise auf der JVM?",
]

for query in RAG_CASES:
    report = rag_with_check(query)
    print(f"Frage: {query}")
    print(f"Dokumente: {', '.join(report['doc_ids'])}")
    print(f"Antwort: {report['answer']}")
    print(f"Faithfulness: {report['validity']}\n")

## Abgabe und Reflexion
**Pflichtabgabe:**
1. Dokumentiere die Top-3-Trefferquote fuer Vektorsuche und Hybridsuche.
2. Gib fuer jede RAG-Frage die verwendeten Dokument-IDs und das Faithfulness-Label an.
3. Begruende in 3 bis 5 Saetzen, wann der hybride Schritt gegenueber reiner Vektorsuche geholfen hat.

**Transferaufgaben:**
1. Filtere bei der Suche nur nach dem Thema `ml`. Wie veraendert sich die Trefferquote?
2. Erweitere den Korpus um 5 eigene Dokumente und fuege mindestens ein neues Metadatenfeld hinzu.
3. Formuliere eine Frage, die absichtlich nicht beantwortbar ist. Reagiert dein RAG-System sauber mit `Unbekannt`?